In [ ]:
import sys
from pathlib import Path

# Add project root to Python path
sys.path.append(str(Path.cwd().parent))

from fit_training_set import *

In [ ]:
sampling_frequency = 2048 # or 4096
duration = 4 # seconds
time_array = np.linspace(-duration, 0, int(sampling_frequency * duration))  # time in seconds

gt = Generate_Offline_Surrogate(time_array=time_array, 
                                ecc_ref_parameterspace=np.linspace(0.001, 0.1, num=3),
                                mean_ano_parameterspace=np.linspace(0, 2*np.pi, num=3),
                                mass_ratio_parameterspace=[1],
                                chi1_parameterspace=[0],
                                chi2_parameterspace=[0],
                                M_output_wfs_per_dimension=500, 
                                min_greedy_error_amp=1e-8,
                                min_greedy_error_phase=1e-6,
                                minimum_spacing_greedy=0.003, 
                                training_set_selection='greedy')


train_obj_p = gt.get_training_set_greedy(property='phase', 
                           min_greedy_error=1e-8, 
                           plot_greedy_error=True,
                           plot_training_set=True,
                           plot_emp_nodes_on_basis=True
                           )

# train_obj_a = gt.get_training_set_greedy(property='amplitude',
#                            min_greedy_error=1e-6,
#                            plot_greedy_error=True,
#                            plot_training_set=True,      
#                            plot_emp_nodes_on_basis=True
#                            )

In [ ]:
def _gaussian_process_regression_t(
        self,
        time_node,
        train_obj: TrainingSetResults,
        optimized_kernel=None,
        plot_kernels=False,
        save_fig_kernels=False,
        time_coupled=False
    ):
    """
    Gaussian Process Regression for one empirical time node.

    ------------------------------------------------------------------
    WHAT THIS FUNCTION DOES
    ------------------------------------------------------------------

    We want to learn:

        f(parameters) -> waveform quantity

    where the waveform quantity is either:
        - phase residual
        - amplitude residual

    at ONE empirical time node.

    ------------------------------------------------------------
    TWO MODES
    ------------------------------------------------------------

    1) time_coupled = False
       --------------------------------
       Each time node gets its own independent GP.

       Input dimensions:
           [e, l, q, chi1, chi2]

       This is:
           - simpler
           - faster
           - usually more stable
           - easier to optimize

       Recommended FIRST approach.


    2) time_coupled = True
       --------------------------------
       Time is added as another GP dimension.

       Input dimensions:
           [e, l, q, chi1, chi2, t]

       The GP then learns correlations across time.

       Advantages:
           - smoother evolution in time
           - fewer discontinuities between nodes

       Disadvantages:
           - MUCH harder optimization
           - higher dimensionality
           - requires more training data
           - more prone to over-smoothing

       Recommended only after the uncoupled model works well.

    ------------------------------------------------------------------
    KERNEL SEARCH STRATEGY
    ------------------------------------------------------------------

    We test MANY kernels automatically.

    We vary:
        - kernel smoothness (nu)
        - length scales
        - isotropic vs anisotropic kernels
        - noise levels

    Then we select the BEST kernel using:

        score = LML - alpha * train_rmse

    because:

        LML alone often prefers overly smooth kernels.

    Adding RMSE penalizes kernels that fit poorly.

    ------------------------------------------------------------------
    IMPORTANT KERNEL CONCEPTS
    ------------------------------------------------------------------

    Matern kernel:
        Controls smoothness of interpolation.

    nu:
        0.5  -> rough / jagged
        1.5  -> moderately smooth
        2.5  -> very smooth

    length_scale:
        Controls correlation distance.

        small:
            rapid variation

        large:
            smooth variation

    isotropic:
        one length scale for ALL dimensions

    anisotropic:
        separate length scale per dimension

        VERY important in high dimensions because:
            eccentricity may vary rapidly
            while spin dependence may vary slowly

    WhiteKernel:
        models numerical noise.

    ------------------------------------------------------------------
    HOW TO TEST EFFICIENTLY
    ------------------------------------------------------------------

    STEP 1:
        Start SIMPLE:

            time_coupled = False

            smoothness_params = [1.5]
            ls_multipliers = [1.0]
            noise_levels = [1e-6]

    STEP 2:
        Inspect:
            - train_rmse
            - plots
            - prediction smoothness

    STEP 3:
        Add anisotropic kernels.

        This is usually the BIGGEST improvement
        for multidimensional waveform spaces.

    STEP 4:
        Add more:
            smoothness_params
            ls_multipliers

    STEP 5:
        ONLY THEN try:
            time_coupled=True

    ------------------------------------------------------------------
    PRACTICAL ADVICE
    ------------------------------------------------------------------

    In high dimensions:

        anisotropic kernels > isotropic kernels

    almost always.

    Also:

        too many optimizer restarts
        can dominate runtime.

    Start with:
        n_restarts_optimizer=5

    before using:
        20

    ------------------------------------------------------------------
    """

    # ============================================================
    # BUILD INPUT MATRICES
    # ============================================================

    if time_coupled is False:

        # --------------------------------------------------------
        # STANDARD MODE:
        # one GP per time node
        # --------------------------------------------------------

        X = np.asarray(self.parameter_grid)

        X_train = X[train_obj.basis_indices]

    else:

        # --------------------------------------------------------
        # TIME-COUPLED MODE:
        # append time as extra dimension
        # --------------------------------------------------------

        X_param = np.asarray(self.parameter_grid)

        t = np.full((X_param.shape[0], 1), self.time[time_node])

        X_train = np.hstack([
            X_param[train_obj.basis_indices],
            t[train_obj.basis_indices]
        ])

        X = np.hstack([
            X_param,
            t
        ])

    # ============================================================
    # TRAINING TARGETS
    # ============================================================

    y_train = np.squeeze(train_obj.training_set.T[time_node])

    # ============================================================
    # SCALE INPUTS
    # ============================================================

    # ------------------------------------------------------------
    # WHY SCALE?
    #
    # GP kernels are VERY sensitive to feature scales.
    #
    # Example:
    #
    #     eccentricity ~ 0.01
    #     q            ~ 10
    #
    # Without scaling:
    #     q dominates distance calculations.
    # ------------------------------------------------------------

    scaler_x = StandardScaler()

    X_train_scaled = scaler_x.fit_transform(X_train)

    X_scaled = scaler_x.transform(X)

    scaler_y = StandardScaler()

    y_train_scaled = scaler_y.fit_transform(
        y_train.reshape(-1, 1)
    ).flatten()

    # ============================================================
    # ESTIMATE CHARACTERISTIC LENGTH SCALE
    # ============================================================

    # ------------------------------------------------------------
    # We estimate a reasonable initial kernel size from
    # nearest-neighbor distances.
    #
    # This gives the optimizer a MUCH better starting point.
    # ------------------------------------------------------------

    nn = NearestNeighbors(n_neighbors=2)

    nn.fit(X_train_scaled)

    distances, indices = nn.kneighbors(X_train_scaled)

    median_nn_distance = np.median(distances[:, 1])

    base_ls = max(0.1, median_nn_distance * 2)

    print("base_ls =", base_ls)

    # ============================================================
    # INPUT DIMENSION
    # ============================================================

    dim = X_train_scaled.shape[1]

    # ============================================================
    # KERNEL SEARCH PARAMETERS
    # ============================================================

    # ------------------------------------------------------------
    # Multipliers applied to base_ls.
    #
    # small:
    #     rapid local variation
    #
    # large:
    #     smoother interpolation
    # ------------------------------------------------------------

    ls_multipliers = [0.3, 1.0, 3.0, 10.0]

    # ------------------------------------------------------------
    # Upper optimization bounds for length scales.
    #
    # Prevents optimizer from going to absurdly smooth kernels.
    # ------------------------------------------------------------

    ls_upper_bounds = [0.5, 1.0, 2.0, 5.0]

    # ------------------------------------------------------------
    # Matern smoothness values.
    #
    # lower:
    #     rougher functions
    #
    # higher:
    #     smoother functions
    # ------------------------------------------------------------

    smoothness_params = [0.5, 1.5, 2.5]

    kernels = []

    # ============================================================
    # 1. ISOTROPIC MATERN KERNELS
    # ============================================================

    # ------------------------------------------------------------
    # One single length scale shared across ALL dimensions.
    #
    # Simpler.
    # Faster.
    # Often insufficient in high dimensions.
    # ------------------------------------------------------------

    for nu in smoothness_params:

        for ls_mult in ls_multipliers:

            for ls_ub in ls_upper_bounds:

                ls = base_ls * ls_mult

                if ls <= ls_ub:

                    kernel = Matern(
                        length_scale=ls,
                        length_scale_bounds=(1e-2, ls_ub),
                        nu=nu
                    )

                    kernels.append({
                        "kernel": kernel,
                        "type": "isotropic",
                        "nu": nu,
                        "ls_ub": ls_ub
                    })

    # ============================================================
    # 2. ANISOTROPIC MATERN KERNELS
    # ============================================================

    # ------------------------------------------------------------
    # Separate length scale per dimension.
    #
    # MUCH more powerful for:
    #     [e, l, q, chi1, chi2]
    #
    # because every parameter may vary differently.
    # ------------------------------------------------------------

    for nu in smoothness_params:

        for ls_mult in ls_multipliers:

            for ls_ub in ls_upper_bounds:

                ls = base_ls * ls_mult

                if ls <= ls_ub:

                    kernel = Matern(
                        length_scale=np.ones(dim) * ls,
                        length_scale_bounds=(1e-2, ls_ub),
                        nu=nu
                    )

                    kernels.append({
                        "kernel": kernel,
                        "type": "anisotropic",
                        "nu": nu,
                        "ls_ub": ls_ub
                    })

    # ============================================================
    # 3. ADD WHITE NOISE TERMS
    # ============================================================

    # ------------------------------------------------------------
    # Models numerical noise / imperfect data.
    #
    # Prevents overfitting.
    # ------------------------------------------------------------

    noise_levels = [1e-8, 1e-6, 1e-4]

    extra_kernels = []

    for entry in kernels:

        for noise in noise_levels:

            noisy_kernel = (
                entry["kernel"]
                + WhiteKernel(
                    noise_level=noise,
                    noise_level_bounds=(1e-10, 1e-1)
                )
            )

            extra_kernels.append({
                "kernel": noisy_kernel,
                "type": entry["type"] + "_noise",
                "nu": entry["nu"],
                "ls_ub": entry["ls_ub"],
                "noise": noise
            })

    kernels.extend(extra_kernels)

    print(f"Generated {len(kernels)} kernels")

    # ============================================================
    # STORAGE
    # ============================================================

    y_predict_kernels = []
    y_predict_std_kernels = []

    best_score = -np.inf
    best_lml = -np.inf
    best_train_rmse = np.inf

    # ------------------------------------------------------------
    # IMPORTANT:
    #
    # LML alone tends to prefer over-smoothed kernels.
    #
    # We therefore penalize high training error.
    #
    # Larger alpha:
    #     prioritize fit quality more
    #
    # Smaller alpha:
    #     prioritize smoothness / generalization more
    # ------------------------------------------------------------

    alpha = 10.0

    # ============================================================
    # KERNEL SEARCH
    # ============================================================

    for entry in kernels:

        try:

            start = time.time()

            gaussian_process = GaussianProcessRegressor(
                kernel=entry["kernel"],
                n_restarts_optimizer=20,
                random_state=42
            )

            # ----------------------------------------------------
            # TRAIN GP
            # ----------------------------------------------------

            gaussian_process.fit(
                X_train_scaled,
                y_train_scaled
            )

            optimized_kernel = gaussian_process.kernel_

            end = time.time()

            # ----------------------------------------------------
            # LOG MARGINAL LIKELIHOOD
            # ----------------------------------------------------

            lml = gaussian_process.log_marginal_likelihood_value_

            # ----------------------------------------------------
            # TRAINING ERROR
            # ----------------------------------------------------

            y_pred_train, std_train = gaussian_process.predict(
                X_train_scaled,
                return_std=True
            )

            train_rmse = np.sqrt(
                np.mean(
                    (y_pred_train - y_train_scaled) ** 2
                )
            )

            # ----------------------------------------------------
            # COMBINED SCORE
            # ----------------------------------------------------

            score = lml - alpha * train_rmse

            # ----------------------------------------------------
            # PREDICT FULL GRID
            # ----------------------------------------------------

            y_predict_scaled, std_prediction_scaled = (
                gaussian_process.predict(
                    X_scaled,
                    return_std=True
                )
            )

            y_predict = scaler_y.inverse_transform(
                y_predict_scaled.reshape(-1, 1)
            ).flatten()

            y_predict_std = (
                std_prediction_scaled
                * scaler_y.scale_[0]
            )

            y_predict_kernels.append(y_predict)

            y_predict_std_kernels.append(y_predict_std)

            self._diagnose_gpr_issues(
                gaussian_process,
                X_train_scaled,
                y_train_scaled
            )

            # ----------------------------------------------------
            # KEEP BEST KERNEL
            # ----------------------------------------------------

            if score > best_score:

                best_score = score

                best_lml = lml

                best_train_rmse = train_rmse

                best_guess_kernel = entry

                best_optimized_kernel = optimized_kernel

                best_y_predict = y_predict

                best_y_predict_std = y_predict_std

        except Exception as e:

            print(
                self.colored_text(
                    f"GPR failed for kernel {entry}: {e}",
                    "red"
                )
            )

            traceback.print_exc()

            continue

    # ============================================================
    # FINAL REPORT
    # ============================================================

    print(
        self.colored_text(
            f"Best kernel = {best_optimized_kernel}\n"
            f"LML = {best_lml:.4f}\n"
            f"RMSE = {best_train_rmse:.6e}\n"
            f"Score = {best_score:.4f}",
            "green"
        )
    )

    # ============================================================
    # RETURN
    # ============================================================

    return (
        best_y_predict,
        [
            best_y_predict - 1.96 * best_y_predict_std,
            best_y_predict + 1.96 * best_y_predict_std
        ],
        best_optimized_kernel,
        best_lml
    )